# 02 · 映射:怎么把一栋楼对应到"一个权利方"
上一本把多源数据拼成了一张表;**这本讲那张表怎么变成「谁算谁的」**。
方法刻意简单:不做语意推断、不算概率,就一张**离散、可读、可改**的查表(`shanghai_lookup.yaml`)。

## 怎么执行
- 点一格,按 **Shift+Enter** 执行;或选单 Run → Run All Cells 从头跑。
- 结果与图会显示在该格下面。代码都在 `engine/` 文件夹,平时不用打开。

In [ ]:
# 让 notebook 找到 engine 里的代码(这格不用改,直接执行)
import sys, os
sys.path.insert(0, os.path.abspath("engine"))
import importlib, config; importlib.reload(config)
import common as C, plots
print("就绪 ·  当前站点 SLUG =", config.SLUG, "(", config.site_name(), ")")

In [ ]:
df = C.current_buildings(config.SLUG)   # 读当前站点(随包缓存或你建的)
print("载入 %d 栋  ·  %s" % (len(df), config.site_name()))

## 一、规则:一栋 = 一个 stakeholder,第一个来源命中即定
5 个权利方:`state` 政府/公共 · `developer` 开发商/资本 · `resident` 居民 ·
`informal` 非正式(本数据无信号、恒空) · `unknown` 未标。

**级联 cascade**:按 `euluc → function → aoi` 顺序问,**第一个给出映射的来源就定**,后面的不再问;
都不中 = `unknown`。`informal` 不在表里(数据无信号),所以永远不会被指派——这是诚实,不是遗漏。

In [ ]:
lk = C.load_lookup()        # 读 shanghai_lookup.yaml
print("级联顺序:", lk["cascade"])
print("\nEULUC 用途 → 权利方(主键,政商可分):")
for k, v in lk["euluc"].items():
    print("   %-10s -> %s" % (k, v))
print("\n默认(都不中):", lk["default"], " · informal 不在表中(数据无信号 → 恒空)")

## 二、走查:一栋楼是怎么被判定的
看几栋楼的多源字段,跟着 `assign_stakeholder` 走一遍——你会看到它**在哪一级命中**。

In [ ]:
lk = C.load_lookup()
cols = [c for c in ["euluc","function","aoi_type2","aoi_type1"] if c in df.columns]
sample = df.head(6)
for _, r in sample.iterrows():
    fields = {c: C._t(r.get(c)) for c in cols}
    print("%-22s -> %s" % (str(fields), C.assign_stakeholder(r, lk)))

## 三、改这张表 = 改"谁算谁的"(反事实的一半)
学生改 `shanghai_lookup.yaml`,就是改权力地图。下面**不动文件**、在内存里试一条改动:
把工业用地从「开发商」改判给「政府」,看分布怎么变。满意了再去真正改 YAML。

In [ ]:
import copy
lk2 = copy.deepcopy(C.load_lookup())
# 找到 euluc 里第一条映射到 developer 的用途,改判给 state(示范一次反事实)
flip = next((k for k, v in lk2["euluc"].items() if v == "developer"), None)
lk2["euluc"][flip] = "state"
a0 = C.assign_all(df).stakeholder.value_counts().to_dict()
a1 = C.assign_all(df, lk2).stakeholder.value_counts().to_dict()
print("把『%s』从 developer 改判给 state:" % flip)
print("  改前:", a0)
print("  改后:", a1)

## 四、贴出来看:权力地图
把每栋楼按权利方著色——这就是 step2 的「权力地图」,后面只调高度 / 上算子都建立在它上面。

In [ ]:
df = C.assign_all(df)
print("各权利方栋数:", df.stakeholder.value_counts().to_dict())
plots.power_map(df)     # 左:footprint 按角色著色;右:栋数 / 面积占比

## 诚实边界(映射这一关)
- `unknown` 是没 join 到用途的楼,**不硬猜**产权。
- EULUC 面级优先 → 居住地块里的零星公建被并入「居民」(简化)。
- `developer` 这里指「资本/商业开发」的用途口径,**不是产权考证**。
- 这是 **forward 教学练习**(改权力 → 看形态),不是规划预测。

> 下一本:[`03 城市天际线-动手做`] —— 锁死 footprint 与标签,只放开高度,看权力怎么改写天际线。